In [1]:
from huggingface_hub import login
login()  # <-- paste your HF token when prompted

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Fine-tuning on the Kotak Dataset

In [5]:
!pip install diffusers transformers torch peft gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [4]:
import os
import json
import csv
import zipfile
from math import ceil
import gdown

# Install gdown to download files from Google Drive
!pip install gdown

def download_and_extract(url, dest_zip, extract_to):
    """Download the dataset and extract the zip file."""
    if not os.path.exists(dest_zip):
        print(f"Downloading from {url} to {dest_zip}...")
        gdown.download(url, dest_zip, quiet=False)
        print("Download complete.")
    else:
        print(f"{dest_zip} already exists, skipping download.")

    print(f"Extracting {dest_zip} to {extract_to}...")
    with zipfile.ZipFile(dest_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete.")

def build_annotations_from_images(base_image_dir, output_json):
    """Build COCO-style annotations from image-text pairs."""
    valid_extensions = [".jpg", ".jpeg", ".png"]
    image_id = 1
    images = []
    annotations = []

    for year in ["2021", "2022", "2023"]:
        year_path = os.path.join(base_image_dir, year)
        if not os.path.isdir(year_path):
            continue

        for fname in os.listdir(year_path):
            img_path = os.path.join(year_path, fname)
            if not os.path.isfile(img_path):
                continue

            name, ext = os.path.splitext(fname)
            if ext.lower() not in valid_extensions:
                continue

            txt_file = os.path.join(year_path, f"{name}.txt")
            if not os.path.isfile(txt_file):
                continue

            with open(txt_file, 'r', encoding='utf-8') as tf:
                caption = tf.read().strip()

            images.append({
                "id": image_id,
                "file_name": os.path.relpath(img_path, base_image_dir)
            })
            annotations.append({
                "image_id": image_id,
                "caption": caption
            })
            image_id += 1

    coco_json = {"images": images, "annotations": annotations}
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(coco_json, f, indent=4)

    print(f"Saved COCO-style annotations to {output_json}")
    return output_json

def prepare_csv_batches(input_json, img_dir, output_dir, batch_size=200):
    """Prepare CSV batches for training."""
    with open(input_json, 'r', encoding="utf-8") as f:
        data = json.load(f)

    image_id_to_filename = {img["id"]: img["file_name"] for img in data["images"]}
    image_id_to_caption = {ann["image_id"]: ann["caption"] for ann in data["annotations"]}

    rows = []
    for img_id, file_name in image_id_to_filename.items():
        full_path = os.path.join(img_dir, file_name)
        if os.path.isfile(full_path):
            rows.append({
                "image_path": full_path,
                "prompt": image_id_to_caption[img_id]
            })

    os.makedirs(output_dir, exist_ok=True)
    total_entries = len(rows)
    num_batches = ceil(total_entries / batch_size)

    print(f"Preparing {num_batches} CSV batch files...")
    for batch_idx in range(num_batches):
        batch_rows = rows[batch_idx * batch_size: (batch_idx + 1) * batch_size]
        output_file = os.path.join(output_dir, f"batch_{batch_idx + 1}.csv")
        with open(output_file, mode='w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=["image_path", "prompt"])
            writer.writeheader()
            writer.writerows(batch_rows)
        print(f"Wrote {len(batch_rows)} entries to {output_file}")

def generalize_dataset_pipeline(
    base_dir="kotakb",
    dataset_url=None,
    zip_filename="dataset.zip",
    dataset_folder="kotakb_21-23_with-captions_tagged",
    batch_size=200
):
    """Complete pipeline to download, extract, convert, and batch the dataset."""
    os.makedirs(base_dir, exist_ok=True)
    dataset_zip = os.path.join(base_dir, zip_filename)

    # Download & extract
    if dataset_url is not None:
        download_and_extract(dataset_url, dataset_zip, base_dir)

    full_dataset_path = os.path.join(base_dir, dataset_folder)
    annotations_path = os.path.join(full_dataset_path, "annotations.json")

    # Build COCO-style JSON annotations
    build_annotations_from_images(full_dataset_path, annotations_path)

    # Prepare CSV batches
    prepare_csv_batches(
        input_json=annotations_path,
        img_dir=full_dataset_path,
        output_dir=os.path.join(base_dir, "csv_batches"),
        batch_size=batch_size
    )

# Run the pipeline in Google Colab
dataset_url = "https://drive.google.com/uc?export=download&id=1Ifk07HeqxHfCCOCvb5oDF-cdxfkfsuq-"  # Example dataset URL
generalize_dataset_pipeline(dataset_url=dataset_url)


Downloading...
From: https://drive.google.com/uc?export=download&id=1Ifk07HeqxHfCCOCvb5oDF-cdxfkfsuq-
To: /content/kotakb/dataset.zip
100%|██████████| 20.9M/20.9M [00:00<00:00, 113MB/s] 


Download complete.
Extracting kotakb/dataset.zip to kotakb...
Extraction complete.
Saved COCO-style annotations to kotakb/kotakb_21-23_with-captions_tagged/annotations.json
Preparing 1 CSV batch files...
Wrote 109 entries to kotakb/csv_batches/batch_1.csv


In [5]:
import os
import csv
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
import gc

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CustomImageTextDataset(Dataset):
    def __init__(self, csv_file):
        self.entries = []
        with open(csv_file, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.entries.append(row)

        # Simpler transformations to reduce memory usage
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        image = Image.open(entry["image_path"]).convert("RGB")
        image = self.transform(image)
        prompt = entry["prompt"]
        return {"pixel_values": image, "prompt": prompt}

def train_model(dataset_csv, num_epochs=5, batch_size=1, learning_rate=1e-4):
    # Memory cleanup before starting
    torch.cuda.empty_cache()
    gc.collect()

    # Load the training dataset with a small batch size
    dataset = CustomImageTextDataset(dataset_csv)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=1)  # Reduced workers and batch size

    print("Loading tokenizer...")
    model_id = "CompVis/stable-diffusion-v1-4"
    tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")

    print("Loading text encoder...")
    text_encoder = CLIPTextModel.from_pretrained(
        model_id,
        subfolder="text_encoder",
        torch_dtype=torch.float16
    )
    text_encoder = text_encoder.to(device)
    text_encoder.requires_grad_(False)  # Freeze text encoder

    print("Loading UNet and VAE...")
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
    )

    # Enable gradient checkpointing for memory efficiency
    pipe.unet.enable_gradient_checkpointing()

    # Move VAE and UNet to device; keep VAE in eval mode to save memory
    pipe.vae = pipe.vae.to(device)
    pipe.vae.requires_grad_(False)
    pipe.vae.eval()

    pipe.unet = pipe.unet.to(device)

    # Apply LoRA with minimal parameters
    lora_config = LoraConfig(
        r=4,  # Reduced rank
        lora_alpha=8,
        target_modules=["to_k", "to_v"],
        lora_dropout=0.0,  # Remove dropout to save compute
        bias="none",
    )
    pipe.unet = get_peft_model(pipe.unet, lora_config)
    print("Applied LoRA injection via PEFT.")

    optimizer = torch.optim.AdamW(
        pipe.unet.parameters(),
        lr=learning_rate,
        weight_decay=0.01,
        eps=1e-8
    )

    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

    # Training loop with memory optimizations
    pipe.unet.train()
    total_steps = len(dataloader) * num_epochs
    print(f"Starting training with {total_steps} total steps")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        for step, batch in enumerate(dataloader):
            if step % 5 == 0:
                torch.cuda.empty_cache()

            with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):  # Mixed precision
                # Process text conditioning
                text_inputs = tokenizer(
                    batch["prompt"],
                    padding="max_length",
                    truncation=True,
                    max_length=77,
                    return_tensors="pt"
                )
                text_input_ids = text_inputs.input_ids.to(device)

                with torch.no_grad():
                    encoder_hidden_states = text_encoder(text_input_ids)[0]

                images = batch["pixel_values"].to(device, dtype=torch.float16)

                with torch.no_grad():
                    latents = pipe.vae.encode(images).latent_dist.sample() * 0.18215

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device)

                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                noise_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample

                loss = torch.nn.functional.mse_loss(noise_pred, noise)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(pipe.unet.parameters(), max_norm=1.0)

            optimizer.step()
            optimizer.zero_grad()

            if step % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Step [{step}/{len(dataloader)}] Loss: {loss.item():.4f}")

            del noise_pred, loss, noisy_latents, latents, noise, images, text_input_ids, encoder_hidden_states
            torch.cuda.empty_cache()

    os.makedirs("lora_finetuned", exist_ok=True)
    pipe.unet.save_pretrained("lora_finetuned")
    print("Training complete! LoRA weights saved to 'lora_finetuned'.")

    del pipe, text_encoder, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

def generate_image(prompt, output_file="generated_image.png", num_inference_steps=25, guidance_scale=7.5):
    torch.cuda.empty_cache()
    gc.collect()

    # Prepare output filenames for base and fine-tuned images.
    base_output = output_file.replace(".png", "_base.png")
    fine_tuned_output = output_file.replace(".png", "_fine_tuned.png")
    model_id = "CompVis/stable-diffusion-v1-4"

    # Generate image using the base pipeline
    print("Generating image with the base model...")
    base_pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    )
    base_pipe = base_pipe.to(device)
    with torch.inference_mode():
        base_image = base_pipe(
            prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            height=512,
            width=512,
        ).images[0]
    base_image.save(base_output)
    print(f"Base image saved to {base_output}.")

    # Generate image using the fine-tuned pipeline (if available)
    print("Generating image with the fine-tuned model...")
    fine_pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    )
    fine_pipe = fine_pipe.to(device)
    if os.path.exists("lora_finetuned"):
        try:
            if hasattr(fine_pipe.unet, 'load_adapter'):
                fine_pipe.unet.load_adapter("lora_finetuned")
                print("Loaded fine-tuned LoRA weights via adapter.")
            else:
                state_dict = torch.load(os.path.join("lora_finetuned", "pytorch_model.bin"), map_location=device)
                fine_pipe.unet.load_state_dict(state_dict, strict=False)
                print("Loaded fine-tuned LoRA weights via state dict.")
        except Exception as e:
            print(f"Fine-tuned weights not loaded, using base model. Error: {e}")
    else:
        print("No fine-tuned weights found. Using base model for fine-tuned image generation.")

    with torch.inference_mode():
        fine_image = fine_pipe(
            prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            height=512,
            width=512,
        ).images[0]
    fine_image.save(fine_tuned_output)
    print(f"Fine-tuned image saved to {fine_tuned_output}.")

    del base_pipe, fine_pipe
    torch.cuda.empty_cache()
    gc.collect()

def run_stable_diffusion(mode, input_data, epochs=3, batch_size=1, lr=5e-5, output_file="generated_image.png"):
    """
    Runs either training or image generation.

    Parameters:
    - mode: "t" for training, "r" for running/inference.
    - input_data: For training mode, path to dataset CSV; for inference mode, the text prompt.
    - epochs: Number of training epochs (default: 3).
    - batch_size: Training batch size (default: 1).
    - lr: Learning rate for training (default: 5e-5).
    - output_file: Output filename for generated image (only used in inference mode).
    """
    if mode == "t":
        if not os.path.exists(input_data):
            print(f"Dataset file {input_data} not found!")
            return
        train_model(
            dataset_csv=input_data,
            num_epochs=epochs,
            batch_size=batch_size,
            learning_rate=lr
        )
    elif mode == "r":
        generate_image(prompt=input_data, output_file=output_file)
    else:
        print("Invalid mode. Use 't' for training or 'r' for running/inference.")


## Run model

In [11]:
# Example usage:

# For training (ensure you provide a valid CSV file path):
run_stable_diffusion(mode="t", input_data="/content/kotakb/csv_batches/batch_1.csv", epochs=50, batch_size=2, lr=5e-5)
#
# For inference (generating an image based on a prompt):
run_stable_diffusion(mode="r", input_data="A girl in the park", output_file="my_generated_image.png")


Loading tokenizer...
Loading text encoder...
Loading UNet and VAE...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Applied LoRA injection via PEFT.
Starting training with 2750 total steps
Epoch 1/50


<ipython-input-5-963d6153d61e>:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):  # Mixed precision


Epoch [1/50] Step [0/55] Loss: 0.1296
Epoch [1/50] Step [10/55] Loss: 0.3025
Epoch [1/50] Step [20/55] Loss: 0.2588
Epoch [1/50] Step [30/55] Loss: 0.1763
Epoch [1/50] Step [40/55] Loss: 0.1628
Epoch [1/50] Step [50/55] Loss: 0.4104
Epoch 2/50
Epoch [2/50] Step [0/55] Loss: 0.1466
Epoch [2/50] Step [10/55] Loss: 0.0529
Epoch [2/50] Step [20/55] Loss: 0.1112
Epoch [2/50] Step [30/55] Loss: 0.4260
Epoch [2/50] Step [40/55] Loss: 0.2263
Epoch [2/50] Step [50/55] Loss: 0.2297
Epoch 3/50
Epoch [3/50] Step [0/55] Loss: 0.1887
Epoch [3/50] Step [10/55] Loss: 0.1329
Epoch [3/50] Step [20/55] Loss: 0.3825
Epoch [3/50] Step [30/55] Loss: 0.3446
Epoch [3/50] Step [40/55] Loss: 0.1853
Epoch [3/50] Step [50/55] Loss: 0.2737
Epoch 4/50
Epoch [4/50] Step [0/55] Loss: 0.1564
Epoch [4/50] Step [10/55] Loss: 0.2763
Epoch [4/50] Step [20/55] Loss: 0.1141
Epoch [4/50] Step [30/55] Loss: 0.2930
Epoch [4/50] Step [40/55] Loss: 0.0349
Epoch [4/50] Step [50/55] Loss: 0.1833
Epoch 5/50
Epoch [5/50] Step [0/55]

...

Base image saved to my_generated_image_base.png.
Generating image with the fine-tuned model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Fine-tuned weights not loaded, using base model. Error: [Errno 2] No such file or directory: 'lora_finetuned/pytorch_model.bin'


  0%|          | 0/25 [00:00<?, ?it/s]

Fine-tuned image saved to my_generated_image_fine_tuned.png.
